# PR1 — Dataset Profiling & Quality Assessment; Data Dictionary & Business Context Mapping

## 01 — Data Profiling & Cleaning
- Purpose: Profile all five raw StreakForge datasets, assess data quality, and build a first-pass data dictionary before any cleaning happens.
- Inputs: data/raw/members_master.csv, subscription_renewal_records.csv, streak_history_episodes.csv, gym_checkin_workout_logs.csv, app_engagement_events.csv
- Outputs: Console/markdown profiling report, docs/data_dictionary.md (first draft)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_DIR = Path("../data/raw")

In [2]:
members  = pd.read_csv(RAW_DIR / "members_master.csv")
subs     = pd.read_csv(RAW_DIR / "subscription_renewal_records.csv")
streak   = pd.read_csv(RAW_DIR / "streak_history_episodes.csv")
checkin  = pd.read_csv(RAW_DIR / "gym_checkin_workout_logs.csv")
app      = pd.read_csv(RAW_DIR / "app_engagement_events.csv")

In [3]:
datasets = {
    "members_master": members,
    "subscription_renewal_records": subs,
    "streak_history_episodes": streak,
    "gym_checkin_workout_logs": checkin,
    "app_engagement_events": app,
}

## 1.1 Business context per source
- members_master — grain: one row per member. Master identity + demographic/profile data.

- subscription_renewal_records — grain: one row per billing cycle. A loyal member has many rows.

- streak_history_episodes — grain: one row per streak segment (not per day).

- gym_checkin_workout_logs — grain: one row per physical check-in.

- app_engagement_events — grain: one row per digital app event.

- gym_checkin_workout_logs + app_engagement_events will later be unioned into a single

- engagement_events fact table (per PRD 5), so their schemas are profiled with that merge in mind.

In [4]:
def profile_dataset(name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Column-level profiling: dtype, nulls, cardinality, sample values."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "n_missing": s.isna().sum(),
            "pct_missing": round(s.isna().mean() * 100, 2),
            "n_unique": s.nunique(dropna=True),
            "sample_values": s.dropna().unique()[:3].tolist(),
        })
    report = pd.DataFrame(rows)
    print(f"\n{'='*90}\n{name}  |  shape={df.shape}  |  memory={df.memory_usage(deep=True).sum()/1e6:.2f} MB\n{'='*90}")
    print(report.to_string(index=False))
    return report

profiles = {name: profile_dataset(name, df) for name, df in datasets.items()}


members_master  |  shape=(17500, 17)  |  memory=16.21 MB
                column  dtype  n_missing  pct_missing  n_unique                                     sample_values
             member_id object          0         0.00     17500              [MBR-003327, MBR-009827, MBR-004184]
             full_name object          0         0.00      4132          [Tarun Singh, Priya Iyengar, Meena Iyer]
                gender object          0         0.00         3                             [Male, Female, Other]
         date_of_birth object        540         3.09     10276              [26-01-2000, 21-07-1994, 2009/09/26]
                  city object          0         0.00        81                      [Bangalore, Chennai, Mumbai]
             city_tier object          0         0.00         3                             [Tier1, Tier2, Tier3]
                 state object          0         0.00        17              [Karnataka, Tamil Nadu, Maharashtra]
            occupation object 

## 1.2 Quality assessment summary
Surfaced from the profiling pass above (confirmed against the raw files):

- members_master.date_of_birth — 540 nulls (~3.1%); also mixed date formats (DD-MM-YYYY and YYYY/MM/DD)
- members_master.join_date — same mixed-format issue, 0 nulls
- members_master.city — 81 raw unique values but only 29 real cities once whitespace is stripped; Bangalore/Bengaluru and Gurgaon/Gurugram are the same city under old/new official names
- subscription_renewal_records — 676 fully duplicated rows; days_before_expiry_renewed has negative values (renewed after expiry — a grace-period signal, not necessarily an error)
- streak_history_episodes.streak_end_date / break_reason — 1,237 nulls each, same rows → structurally null (streak is still ongoing), not missing data
- streak_history_episodes — 25 duplicate rows; streak_length_days has extreme outliers (max 1,683 days)
- gym_checkin_workout_logs.checkout_datetime — 13,144 nulls (forgot-to-checkout pattern);
- trainer_id — 96,099 nulls (self-guided sessions, not an error)
- app_engagement_events.notification_engagement — 72,990 nulls (only applies to notification-type events)

In [6]:
# ---- Duplicate-row check (exact full-row duplicates) ----
for name, df in datasets.items():
    print(f"{name:32s} exact duplicate rows: {df.duplicated().sum()}")

members_master                   exact duplicate rows: 0
subscription_renewal_records     exact duplicate rows: 676
streak_history_episodes          exact duplicate rows: 25
gym_checkin_workout_logs         exact duplicate rows: 0
app_engagement_events            exact duplicate rows: 0


In [7]:
# ---- Referential integrity: does every member_id in the fact tables exist in members_master? ----
member_ids = set(members["member_id"])
for name, df in [("subscription_renewal_records", subs), ("streak_history_episodes", streak),
                  ("gym_checkin_workout_logs", checkin), ("app_engagement_events", app)]:
    orphans = (~df["member_id"].isin(member_ids)).sum()
    print(f"{name:32s} orphan member_id rows: {orphans}")

subscription_renewal_records     orphan member_id rows: 0
streak_history_episodes          orphan member_id rows: 0
gym_checkin_workout_logs         orphan member_id rows: 0
app_engagement_events            orphan member_id rows: 0


In [8]:

# ---- First-pass Data Dictionary draft ----
def build_data_dictionary(profiles: dict) -> pd.DataFrame:
    frames = []
    for name, report in profiles.items():
        r = report.copy()
        r.insert(0, "dataset", name)
        frames.append(r)
    return pd.concat(frames, ignore_index=True)

data_dictionary = build_data_dictionary(profiles)
Path("../docs").mkdir(exist_ok=True)
data_dictionary.to_csv("../docs/data_dictionary_draft.csv", index=False)
print("Draft data dictionary written to docs/data_dictionary_draft.csv —", data_dictionary.shape)

Draft data dictionary written to docs/data_dictionary_draft.csv — (53, 7)
